# 1. 模型结构

- 两个知识点
    - Module
        - Sequential
        - ModuleDict
        - ModuleList
    - Tensor

- 加载模型
    - 用户往往不知道具体的算法模型，transformers框架提供一组Auto类，来实现自动识别。 

In [1]:
from transformers import AutoModel, AutoConfig, AutoImageProcessor, YolosForObjectDetection

# model = AutoModel.from_pretrained("F:/03Models/yolos-tiny")
model = YolosForObjectDetection.from_pretrained("F:/03Models/yolos-tiny")  # 与上一句等价
model

W0326 09:32:00.336000 26988 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of YolosModel were not initialized from the model checkpoint at F:/03Models/yolos-tiny and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


YolosModel(
  (embeddings): YolosEmbeddings(
    (patch_embeddings): YolosPatchEmbeddings(
      (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
    (interpolation): InterpolateInitialPositionEmbeddings()
  )
  (encoder): YolosEncoder(
    (layer): ModuleList(
      (0-11): 12 x YolosLayer(
        (attention): YolosAttention(
          (attention): YolosSelfAttention(
            (query): Linear(in_features=192, out_features=192, bias=True)
            (key): Linear(in_features=192, out_features=192, bias=True)
            (value): Linear(in_features=192, out_features=192, bias=True)
          )
          (output): YolosSelfOutput(
            (dense): Linear(in_features=192, out_features=192, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): YolosIntermediate(
          (dense): Linear(in_features=192, out_features=768, bias=True)
          (intermediate_

- YolosForObjectDetection（核心实现：网络结构，运算）
    - YolosPreTrainedModel（权重初始化）
        - PreTrainedModel（骨干模型以及相关初始化，量化等）
            - nn.Module + (维护模型的基础：权重数据，设备，训练参数，Graph跟踪)

- (1) Module的to的使用（修改Module）
    - Tensor的to

In [13]:
model.to("cuda")    # model被改变
print(model.device)

cuda:0


In [10]:
import torch
x = torch.tensor([1, 2, 3])
y = x.to("cuda")
print(x.device)
print(y.device)
print(y.dtype)

cpu
cuda:0
torch.int64


In [11]:
# help(model.float)

- (2) 提供一组类型转换函数
    - bfloat16()
    - float()
    - half()
    - double()
    - to()
    - cuda()
    - cpu()
    - xpu()  # intel GPU
    - train()
    - eval()

- (3) compile()
    - 负责把模型转换为硬件编码。提升计算速度（基于GPU）
    - 不会直接使用Module.compile。实际是被torch.complie调用。

- (4) 与模型结构的函数
    - parameters() / named_parameters()
        - 本质是张量：主要给优化器使用。有个类型判断的函数
    - buffers() / named_buffers()
        - 就是张量：所有数据的缓冲。
    - modules() / named_modules()
        - 不是张量：是个模块。
    - children() / named_children()
        - 子模块

In [33]:
params = model.parameters()
nparams = model.named_parameters()

In [34]:
# next(params)
for p in params:   # 生成器的使用协程：节省缓存
    print(type(p))
    break

# ps = list(params)
# ps

<class 'torch.nn.parameter.Parameter'>


In [36]:
next(nparams)
type(next(nparams))

tuple

- 说明：
    - parameters与named_parameters的关系
        - 返回名字

In [56]:
nparams = model.named_parameters()
total = 0 
num_p = 0 
for name, p in nparams:
    # print(name, p.nbytes, p.itemsize, p.element_size())
    # print(type(p))   # 参数与Tensor基本一致，额外增加了属性检测（给优化器使用）
    # break
    total += p.nbytes
    # print(p.shape)    # 计算参数的个数
    num_p += p.numel()
    if name == "embeddings.patch_embeddings.projection":
        p.requires_grad = False
print(total / (1024 * 1024))
print(num_p)
    

24.2578125
6359040


In [57]:
# dir(torch.Tensor)

In [60]:
all_modules = model.named_modules()
for n, m in all_modules:
    print(n)
    print("\t", type(m))
    break


	 <class 'transformers.models.yolos.modeling_yolos.YolosModel'>
